In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [4]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [5]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-3-flash-preview")

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4991.26it/s]


In [8]:
#vector store

from langchain_chroma import Chroma

vectorStore = Chroma.from_documents(documents, embedding)

vectorStore

In [ ]:
vectorStore.similarity_search("cat")

[Document(id='43be3dde-5321-47fd-8ed3-7a1333cbebfd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3cb08ae1-ad18-4e7b-ad93-9406aa07d13a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='10dd6219-a4fd-462e-bb7a-e7693aa313ed', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='a31ba5a6-1544-49d7-ac0d-a27086f8e5ce', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [11]:
## async query

await vectorStore.asimilarity_search("cat")

[Document(id='43be3dde-5321-47fd-8ed3-7a1333cbebfd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3cb08ae1-ad18-4e7b-ad93-9406aa07d13a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='10dd6219-a4fd-462e-bb7a-e7693aa313ed', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='a31ba5a6-1544-49d7-ac0d-a27086f8e5ce', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [ ]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorStore.similarity_search).bind(k=1)

retriever.batch(["cat","dog"])

[[Document(id='43be3dde-5321-47fd-8ed3-7a1333cbebfd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3cb08ae1-ad18-4e7b-ad93-9406aa07d13a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [14]:
# this one is more preferable...

retriever = vectorStore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='43be3dde-5321-47fd-8ed3-7a1333cbebfd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3cb08ae1-ad18-4e7b-ad93-9406aa07d13a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [15]:
# RAG

# Take the user question, retrieve related documents from the vectore database, put both into a prompt, and send them to llm and let the LLM answer.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain = {"context":retriever, "question":RunnablePassthrough()}|prompt|model

response = rag_chain.invoke("tell me about cats")

print(response.content)


[{'type': 'text', 'text': 'Cats are independent pets that often enjoy their own space.', 'extras': {'signature': 'Ep0FCpoFAQw51sdDn0hxspH8FFZt+06d67qBkRt4In2ZFoXDYKQIz0o1bVDzgP1VAiudsyLynVeRE350FE27tjUmOUlv/5QRtzO6x+UY1Ukrx+vc5rTA3zAYMLE9nSKW8QH7ivONOZYSyxgUTmY/Tg7a8+SAYO454uUi3eRkuMtR439RZBacqygNWrnSsH5kDFdMN8wA9sSp6WoALC46HZO3jAmFkp38szUQLpmouZdGvDctq0N2UZj3N2ighWNQy5IuXJfyMcABZ4DAj008iAKoGIYZodvfVaKXk7C/PXVoWpRpeVlyItjsf1Cye/1LhAVsbUufcriKzerB5vJSpX1ZAeeaDcj8TCyiNX7tYK+D3et/Z07l3SFpGFw40kiC+DOBmySYcbKXboYNZC+/AK+++xUpzuEErvcKINxJ6U3oVARbAduBzYbhH1uBLqyywlgagjkMXWLeZz5Q/QB7s7sXUTxN0JDC8rBDlsZHsz6Jc7emVKARbtAu+BXTHm6RynvxiBgMdL+O6g+dpOn6v9gkF7KUf7x9YhLDg8B7nGSjX/WRaHIew2nleFrpGqZFBOpLr8FtgkecJYPWK6FmOnKfgXrllMOpr6DOzqtjnnikmueOJn9d4klbFAfMpB+wXISnqLx+0xkY5hdA2E9u/4oj2GQLoTlEkUb22+KFkPgbWOrqeXkGf9YD8lLFqcxbLi9FQ/d92JI3ygNRnhjbg9mxG+qrNCoQSqJQ1cJW9KZca7hHlUp6g1ciXF8/fp4GySm7Dt4+y/NBjNFHq0rqcUOup6j8Wre90v8vnC6Eby/dkh1mDEUmMhWYqYwFIro2zC+xzreBzsBQ4qDgIONe4wLPTb3Gg8KYuy4bSQ7zC3lSWYut8VUxJM

In [16]:
# Input Question
#       ↓
# "tell me about dogs"

#  ┌────────────────────┐
#  │                    │
#  ↓                    ↓

# Retriever          Passthrough
# (search docs)      (keep question)

#  ↓                    ↓

# "context"          "question"

#  └────────┬───────────┘
#           ↓

#        Prompt

#           ↓

#          LLM

#           ↓

#        Answer
       